# 🎓 Inscora Enterprise — Google Colab Cloud Runner (Free T4 GPU)

Run the full **GLM-OCR Vision + LLaMA 3.2:3B** exam correction suite on Google's cloud with zero battery, CPU, or GPU usage on your laptop.

### ⚡ Setup Instructions:
1. Click **Runtime** > **Change runtime type**
2. Choose **T4 GPU** under Hardware accelerator and click **Save**
3. Run the cells below one-by-one or click **Runtime** > **Run all**


In [ ]:
# Cell 1: Check Tesla T4 GPU
!nvidia-smi


In [ ]:
# Cell 2: Install Ollama Engine & Cloudflare Tunnel binary
!curl -fsSL https://ollama.ai/install.sh | sh
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared


In [ ]:
# Cell 3: Start Ollama in background & pull models (GLM-OCR + LLaMA 3.2)
import subprocess, time
ollama_proc = subprocess.Popen(['ollama', 'serve'])
time.sleep(4)
!ollama pull glm-ocr:latest
!ollama pull llama3.2:3b


In [ ]:
# Cell 4: Clone repository & install Python requirements
!git clone https://github.com/CYRUS-pinto/inscrona.git
%cd inscrona/glm-version
!pip install -q -r requirements.txt


In [ ]:
# Cell 5: Launch FastAPI + Expose via Public Cloudflare URL
import subprocess, time, re, sys

server_proc = subprocess.Popen([sys.executable, '-m', 'uvicorn', 'app.main:app', '--host', '0.0.0.0', '--port', '8001'])
time.sleep(3)

tunnel_proc = subprocess.Popen(['/content/cloudflared', 'tunnel', '--url', 'http://localhost:8001'], stderr=subprocess.PIPE, text=True)

print('Generating secure public tunnel...')
while True:
    line = tunnel_proc.stderr.readline()
    m = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
    if m:
        url = m.group(0)
        print('\n' + '='*75)
        print('🚀 INSCORA IS ONLINE ON GOOGLE COLAB (T4 GPU)!')
        print(f'👉 ACCESS YOUR APP HERE: {url}')
        print('='*75 + '\n')
        break
